# ForgeryNet: Dual-Stream Multi-Task Indic Handwriting Forgery Forensics
### Complete Reviewer Reproduction Notebook

This notebook enables end-to-end replication of all experimental results reported in:
> **ForgeryNet: A Dual-Stream Multi-Task Deep Framework for Detection, Localization, and Categorization of Handwritten Forgeries in Indic Scripts** (*Submitted to Springer IJDAR*)

#### Benchmark Ablation Progression (Table 1):
1. **Stage 1 (Baseline CNN)**: Resized $512\times512$, Unweighted BCE + Dice (Reproduces segmentation collapse)
2. **Stage 2 (Balanced Loss)**: Resized $512\times512$, Focal Tversky Loss ($\alpha=0.7, \beta=0.3$) + Balanced BCE
3. **Stage 3 (Dual-Stream + Attention)**: Resized $512\times512$, RGB + Laplacian noise residual stream + Spatial Self-Attention
4. **Stage 4 (Native Patch SOTA)**: Native-resolution $512\times512$ crops biased on tampering boundaries (89.39% Detection Acc, 0.9446 ROC-AUC, 100% Genuine Specificity, 1.00 Char-Addition F1)

--- 
## 1. Environment Setup & GPU Verification

In [ ]:
# Check CUDA GPU (A100, V100, or T4 recommended)
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

In [ ]:
# Install dependencies
!pip install -q torch torchvision Pillow scikit-learn scikit-image matplotlib tqdm pandas

--- 
## 2. Mount Dataset (Google Drive or Local)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set path to where your Bengali_Handwriting_CopyMove_Dataset is located
DATASET_PATH = '/content/drive/MyDrive/Bengali_Handwriting_CopyMove_Dataset'
import os
print('Dataset directory exists:', os.path.isdir(DATASET_PATH))

--- 
## 3. Clone / Import Codebase

In [ ]:
# If cloning from GitHub:
# !git clone https://github.com/Aptee/ForgeryNet-Indic.git /content/ForgeryNet
# %cd /content/ForgeryNet

import sys
sys.path.insert(0, '/content')
print('Ready to train and evaluate.')

--- 
## 4. Run Any Ablation Experiment (Table 1 Replication)
Select the stage you wish to reproduce:
- `--stage 1`: Baseline CNN (Downsampled $512^2$, unweighted losses)
- `--stage 2`: Balanced Loss (Downsampled $512^2$, Focal Tversky Loss)
- `--stage 3`: Dual-Stream Forensics + Spatial Self-Attention (Downsampled $512^2$)
- `--stage 4`: **Native-Resolution Patch Pipeline (Final Proposed SOTA)**

In [ ]:
# Train Final Proposed SOTA (Stage 4)
!python src/train.py \
    --stage 4 \
    --data_dir "$DATASET_PATH" \
    --out_dir /content/outputs \
    --epochs 35 \
    --bs 8 \
    --lr 1.5e-4

--- 
## 5. Evaluate Best Checkpoint & Reproduce Paper Metrics (Table 1 & Table 2)

In [ ]:
# Evaluate best model on the 132-sample test split
!python src/evaluate.py \
    --checkpoint /content/outputs/stage_4/checkpoints/best.pth \
    --data_dir "$DATASET_PATH" \
    --mode patch \
    --det_thresh 0.32 \
    --seg_thresh 0.25

--- 
## 6. Full-Page Document Visual Inspection (Inference with Bounding Boxes)

In [ ]:
# Run visual forensic inspection on a sample document scan
SAMPLE_IMG = f'{DATASET_PATH}/character_addition/forged/0001_forged.tif'
SAMPLE_MASK = f'{DATASET_PATH}/character_addition/masks/0001_mask.tif'

!python src/inference_full_page.py \
    --image "$SAMPLE_IMG" \
    --mask "$SAMPLE_MASK" \
    --checkpoint /content/outputs/stage_4/checkpoints/best.pth \
    --out forensic_inspection_sample.png

from IPython.display import Image
Image('forensic_inspection_sample.png')